In [1]:
import glob
import random
import os
import time
import json
import re
import pandas as pd
import modified_didppy as m_dp
import gc
import traceback # <--- Added for detailed error logs

# --- CONFIGURATION ---
# Path to the folder containing .txt instance files
DATA_FOLDER_PATH = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\5_JSP_dual_bounds_and_models\Datasets\JSPLIB_instances"

# Path to the instances.json file containing BKS
INSTANCES_JSON_PATH = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\5_JSP_dual_bounds_and_models\Datasets\instances.json"


print("Configuration Loaded.")

Configuration Loaded.


# **Data**

In [ ]:
def parse_jsp_instance(file_content):
    """
    Parses a JSP instance string (Taillard/Beasley/OR-Lib format).
    Format:
    Line 1: num_jobs num_machines
    Lines 2+: machine_index processing_time machine_index processing_time ...
    """
    # 1. Clean and Tokenize
    # Remove comments and split by whitespace
    tokens = []
    for line in file_content.splitlines():
        if not line.strip() or line.strip().startswith(('#', '[')):
            continue
        tokens.extend(re.findall(r'\d+', line))
    
    iterator = iter(tokens)
    
    try:
        num_jobs = int(next(iterator))
        num_machines = int(next(iterator))
    except StopIteration:
        raise ValueError("File is empty or invalid header")

    # 2. Extract Job Data
    # jobs_data format: [ [(machine, time), (machine, time)], ... ]
    jobs_data = []
    for _ in range(num_jobs):
        job_seq = []
        for _ in range(num_machines):
            m = int(next(iterator))
            p = int(next(iterator))
            job_seq.append((m, p))
        jobs_data.append(job_seq)

    num_operations = num_jobs * num_machines

    # 3. Build DIDP Structures (Flattened)
    op_job_type = []
    op_required_machine_type = []
    op_processing_time = []
    op_predecessors = []
    op_deadline = []
    op_same_job = [[] for _ in range(num_operations)] 
    ops_on_machine = [[] for _ in range(num_machines)] 
    map_mj_to_op = {}
    valid_machines = [] # For VM_jo

    # Default deadline (infinity)
    DEFAULT_DEADLINE = float('inf')
    
    op_counter = 0
    for j_idx, job_seq in enumerate(jobs_data):
        job_ops_indices = []
        
        for i, (m, p) in enumerate(job_seq):
            # Attributes
            op_job_type.append(j_idx)
            op_required_machine_type.append(m)
            op_processing_time.append(float(p))
            op_deadline.append(DEFAULT_DEADLINE)
            valid_machines.append([m])
            
            # Map & Sets
            map_mj_to_op[(m, j_idx)] = op_counter
            ops_on_machine[m].append(op_counter)
            job_ops_indices.append(op_counter)
            
            # Predecessors
            if i > 0:
                op_predecessors.append([op_counter - 1])
            else:
                op_predecessors.append([]) 
            
            op_counter += 1
            
        # Same Job Set
        for o_id in job_ops_indices:
            op_same_job[o_id] = [x for x in job_ops_indices if x != o_id]

    # 4. Return Dictionary
    return {
        "metadata": {
            "num_jobs": num_jobs,
            "num_machines": num_machines,
            "num_operations": num_operations
        },
        "DIDP_sets": {
            "op_job_type": op_job_type,
            "op_required_machine_type": op_required_machine_type,
            "op_processing_time": op_processing_time,
            "op_predecessors": op_predecessors,
            "op_deadline": op_deadline,
            "op_same_job": op_same_job,
            "ops_on_machine": ops_on_machine,
            "valid_machines": valid_machines,
            "map_mj_to_op": map_mj_to_op
        }
    }

def load_bks_lookup(json_path):
    """
    Reads the instances.json file and returns a dictionary:
    { 'instance_name': best_known_cost, ... }
    """
    bks_lookup = {}
    
    if not os.path.exists(json_path):
        print(f"Warning: JSON file not found at {json_path}")
        return bks_lookup

    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
            
        # Iterate through the list of instances in the JSON
        for entry in data:
            name = entry.get('name')
            optimum = entry.get('optimum')
            
            # Fallback logic from load_instance.py
            if optimum is None:
                bounds = entry.get('bounds')
                if bounds:
                    optimum = bounds.get('lower') # Or 'upper' if you prefer UB as BKS
            
            # Handle string 'nan' or None
            if optimum == "nan" or optimum is None:
                optimum = None
            else:
                optimum = float(optimum)
                
            if name:
                bks_lookup[name] = optimum
                
        print(f"Loaded BKS for {len(bks_lookup)} instances.")
        
    except Exception as e:
        print(f"Error reading JSON: {e}")
        
    return bks_lookup

# Initialize Global Variables (MUST exist for model function)
current_number_of_operations = 0
current_number_of_machines = 0
current_number_of_jobs = 0
current_op_job_type = []
current_op_required_machine_type = []
current_op_processing_time = []
current_op_deadline = []
current_op_predecessors = []
current_op_same_job = []
current_ops_on_machine = []
current_valid_machines = []
current_map_mj_to_op = {}

# **DIDP model**

In [ ]:
def creation_of_didp_model_function():
    # -----------------------------
    # 1. Retrieve Global Variables
    # -----------------------------
    num_operations = current_number_of_operations
    num_machines = current_number_of_machines
    op_processing_time = current_op_processing_time
    op_deadline = current_op_deadline
    op_predecessors = current_op_predecessors
    op_same_job = current_op_same_job
    valid_machines = current_valid_machines
    ops_on_machine = current_ops_on_machine
    
    # NEW: Added requested variables
    op_job_type = current_op_job_type
    op_required_machine_type = current_op_required_machine_type

    # -----------------------------
    # 2. DIDP Model and Constants
    # -----------------------------
    model = m_dp.Model(maximize=False, float_cost=True)

    operation_obj = model.add_object_type(number=num_operations)
    machine_obj = model.add_object_type(number=num_machines)

    # Constant tables
    processing_time_table = model.add_float_table(op_processing_time)
    deadline_table = model.add_float_table(op_deadline)
    predecessor_table = model.add_set_table(op_predecessors, object_type=operation_obj)
    same_job_table = model.add_set_table(op_same_job, object_type=operation_obj)
    valid_machine_table = model.add_set_table(valid_machines, object_type=operation_obj)

    # -----------------------------
    # 3. State Variables
    # -----------------------------
    unscheduled_operations = model.add_set_var(object_type=operation_obj,
                                    target=list(range(num_operations)))
    finished_operations = model.add_set_var(object_type=operation_obj, target=[])
    machine_available_time = [
        model.add_float_resource_var(target=0, less_is_better=True, name=f"at_{mi}")
        for mi in range(num_machines)
    ]
    op_completion_time = [
        model.add_float_resource_var(target=0, less_is_better=True, name=f"c_{jo}")
        for jo in range(num_operations)
    ]
    jo = model.add_element_var(object_type=operation_obj, target=0, name="jo")
    mi = model.add_element_var(object_type=machine_obj, target=0, name="mi")
    alpha = model.add_int_var(target=0, name="alpha")
    
    # ------------------------------------------------------------
    # 4. Transitions
    # ------------------------------------------------------------  
    # α = 0 transition — select an operation jo to schedule next
    # ------------------------------------------------------------
    for o in range(num_operations):
        select_operation = m_dp.Transition(
            name=f"select_op{o}",
            cost= m_dp.FloatExpr.state_cost(),  # no cost
            preconditions=[
                unscheduled_operations.contains(o),             # jo ∈ U
                predecessor_table[o].issubset(finished_operations),  # pred(jo) ⊆ F
                alpha == 0,                                     # current stage α = 0
            ],
            effects=[
                (jo, o),                                                    # store selected op
                (alpha, 1),                                                 # next stage = 1
            ],
        )
        model.add_transition(select_operation)

    # ------------------------------------------------------------
    # α = 1 transition — assign a machine to the selected operation (jo)
    # ------------------------------------------------------------
    for o in range(num_operations):
        valid_machines_for_op = valid_machines[o]
        expr = m_dp.FloatExpr(0)
        #Finding the completion time of the predecessors to make sure 1 job cannot be operated on 2 machines at the same time
        if op_same_job[o]:
            job_exprs = [op_completion_time[k] for k in op_same_job[o]]
            expr = job_exprs[0]
            for e in job_exprs[1:]:
                expr = m_dp.max(expr, e)
        max_same_job_completion_time = expr
        for m in valid_machines_for_op:
            # Compute start & completion
            start_time = m_dp.max(machine_available_time[m], max_same_job_completion_time)
            completion_time = start_time + processing_time_table[o]
            cost_expr = m_dp.FloatExpr.state_cost()
            transition = m_dp.Transition(
                name=f"schedule_op{o}_on_m{m}",
                cost=cost_expr,
                preconditions=[
                    (jo == o),  # only schedule the chosen operation
                    unscheduled_operations.contains(o),
                    predecessor_table[o].issubset(finished_operations),
                    completion_time <= deadline_table[o],
                    alpha == 1,
                ],
                effects=[
                    (unscheduled_operations, unscheduled_operations.remove(o)),
                    (finished_operations, finished_operations.add(o)),
                    (op_completion_time[o], completion_time),
                    (machine_available_time[m], completion_time),
                    (jo, o),
                    (mi, m),
                    (alpha, 0),  # back to α = 0 after scheduling
                ],
            )
            model.add_transition(transition)

    # ------------------------------------------------------------
    # 5. Base case (OPTIMIZED)
    # ------------------------------------------------------------
    
    # Helper function to build a balanced max tree
    def build_balanced_max(expr_list):
        if not expr_list:
            return m_dp.FloatExpr(0)
        if len(expr_list) == 1:
            return expr_list[0]
        
        # Split in half
        mid = len(expr_list) // 2
        left_expr = build_balanced_max(expr_list[:mid])
        right_expr = build_balanced_max(expr_list[mid:])
        
        return m_dp.max(left_expr, right_expr)

    # Collect all completion time expressions into a list
    all_completion_times = [op_completion_time[o] for o in range(num_operations)]
    
    # Build the balanced expression
    makespan = build_balanced_max(all_completion_times)
    
    model.add_base_case([unscheduled_operations.is_empty()], cost=makespan)
    
    # ------------------------------------------------------------
    # 6. State Constraints
    # ------------------------------------------------------------
    # Enforce 0 <= completion time <= deadline for all operations
    for o in range(num_operations):
        # Upper bound: c(o) <= d(o)
        model.add_state_constr(op_completion_time[o] <= deadline_table[o])

    # ------------------------------------------------------------
    # 7. Dual Bound (optional)
    # ------------------------------------------------------------
    # Machine-based bound: for each machine m, available_time[m] + sum(remaining ptime on that machine)
    ops_on_machine_consts = [
        model.create_set_const(object_type=operation_obj, value=ops) for ops in ops_on_machine
    ]
    # remaining processing time on machine m = sum of ptime for operations on m that are still unscheduled
    remaining_time_on_machine = [
        processing_time_table[unscheduled_operations.intersection(ops_on_machine_consts[m])]
        for m in range(num_machines)
    ]
    machine_bound_exprs = [
        machine_available_time[m] + remaining_time_on_machine[m]
        for m in range(num_machines)
    ]

    # fold to a single dp.max expression
    dual_bound_expr = machine_bound_exprs[0]
    for b in machine_bound_exprs[1:]:
        dual_bound_expr = m_dp.max(dual_bound_expr, b)

    # Add dual bound to model
    model.add_dual_bound(dual_bound_expr)
    #"""

    # =========================================================
    # 8. Bundle Metadata
    # =========================================================
    metadata = {
        "unscheduled_operations": unscheduled_operations,
        "finished_operations": finished_operations,
        "machine_available_time": machine_available_time,
        "op_completion_time": op_completion_time,
        # Added the requested new variables to metadata
        "op_job_type": op_job_type,
        "op_required_machine_type": op_required_machine_type
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# **Selecting instances**

In [13]:
# ==========================================
# USER SETTINGS
# ==========================================

# 1. Define your data folder path (Update this if needed)
# DATA_FOLDER_PATH = r"C:\Users\ACER\Desktop\Code\...\Datasets\JSPLIB_instances"

# 2. LIST FOR YOU TO SPECIFY THE NAMES
# Add the filenames you want to run here.
# If you leave this list EMPTY [], the code will scan the folder and run ALL .txt files.
manual_instances_to_run = [
    "abz5.txt",
    "ft06.txt",
    "ft10.txt",
    "la07.txt",
    "la21.txt",
    
    "la26.txt",
    "ta01.txt",
    "ft20.txt",
    "ta20.txt",
    "ta41.txt"
]

# ==========================================
# GENERATION LOGIC
# ==========================================

instance_names = []

if len(manual_instances_to_run) > 0:
    print(f"Using manual selection of {len(manual_instances_to_run)} instances.")
    instance_names = manual_instances_to_run
else:
    print("Manual list is empty. Scanning folder for all .txt files...")
    if 'DATA_FOLDER_PATH' not in locals():
        print("Error: DATA_FOLDER_PATH is not defined. Please define it or fill the manual list.")
        all_files = []
    else:
        all_files = glob.glob(os.path.join(DATA_FOLDER_PATH, "*.txt"))
        instance_names = [os.path.basename(f) for f in all_files]
        print(f"Found {len(instance_names)} instances in folder.")

# Save to CSV for the next step
df_instances = pd.DataFrame(instance_names, columns=["Instance"])
input_csv_name = "JSP_target_instances.csv" 

df_instances.to_csv(input_csv_name, index=False)

print(f"\nSaved list to: {input_csv_name}")
print(df_instances)

Using manual selection of 10 instances.

Saved list to: JSP_target_instances.csv
   Instance
0  abz5.txt
1  ft06.txt
2  ft10.txt
3  la07.txt
4  la21.txt
5  la26.txt
6  ta01.txt
7  ft20.txt
8  ta20.txt
9  ta41.txt


# **Run single dual bound model on selected instances**

In [17]:
import time as pytime
import csv

# ==========================================
# CONFIGURATION
# ==========================================
input_csv_path = "JSP_target_instances.csv"   # Reads the list you created in Step 1
output_csv_name = "JSP_2T_single_dual_bound_selected_results_1800s_lim.csv"
time_limit_seconds = 1800

bks_map = {}

# Path to your input CSV file
csv_file = "JSP_single_dual_bound_selected_results_10s_lim.csv"

with open(csv_file, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Remove ".txt" from instance name
        instance_name = row["Instance"].replace(".txt", "")
        # Convert Best Known Cost to integer
        best_known_cost = float(row["Best Known Cost"])
        bks_map[instance_name] = best_known_cost

print(bks_map)

# ==========================================
# RESUME LOGIC
# ==========================================
processed_instances = []

if os.path.exists(output_csv_name):
    try:
        df_existing = pd.read_csv(output_csv_name)
        if "Instance" in df_existing.columns:
            processed_instances = df_existing["Instance"].tolist()
        print(f"Found existing results. Resuming... ({len(processed_instances)} instances already done).")
    except:
        pass

# Read the target list
try:
    df_target = pd.read_csv(input_csv_path)
    all_target_instances = df_target["Instance"].tolist()
except Exception as e:
    print(f"Error reading {input_csv_path}: {e}")
    all_target_instances = []

remaining_instances = [inst for inst in all_target_instances if inst not in processed_instances]
print(f"Queued {len(remaining_instances)} instances for execution.")

{'abz5': 1234.0, 'ft06': 55.0, 'ft10': 930.0, 'la07': 890.0, 'la21': 1046.0, 'la26': 1218.0, 'ta01': 1231.0, 'ft20': 1165.0, 'ta20': 1318.0, 'ta41': 1859.0}
Queued 10 instances for execution.


In [18]:
# ==========================================
# EXECUTION LOOP
# ==========================================
for i, instance_name in enumerate(remaining_instances):
    print(f"\n[{i+1}/{len(remaining_instances)}] Processing: {instance_name}")
    
    file_path = os.path.join(DATA_FOLDER_PATH, instance_name)
    
    # Get BKS
    short_name = os.path.splitext(instance_name)[0]
    best_known_cost = bks_map.get(short_name, None) if 'bks_map' in locals() else None
    
    result_entry = {
        "Instance": instance_name,
        "Best Known Cost": best_known_cost,
        "Cost": float('inf'),
        "Gap to BKC": "N/A",
        "Nodes Expanded": 0,
        "Nodes Generated": 0,
        "Running Time (s)": 0,
        "Is Optimal": "Error",
        "Infeasibility": "Error"
    }
    
    try:
        # 1. Parse Data
        with open(file_path, 'r') as f:
            content = f.read()
        parsed_data = parse_jsp_instance(content) # Assumes this function exists from your notebook

        # 2. Update Global Variables
        didp_sets = parsed_data['DIDP_sets']
        current_number_of_operations = parsed_data['metadata']['num_operations']
        current_number_of_machines = parsed_data['metadata']['num_machines']
        current_number_of_jobs = parsed_data['metadata']['num_jobs']
        
        current_op_job_type = didp_sets['op_job_type']
        current_op_required_machine_type = didp_sets['op_required_machine_type']
        current_op_processing_time = didp_sets['op_processing_time']
        current_op_deadline = didp_sets['op_deadline']
        current_op_predecessors = didp_sets['op_predecessors']
        current_op_same_job = didp_sets['op_same_job']
        current_ops_on_machine = didp_sets['ops_on_machine']
        current_valid_machines = didp_sets['valid_machines']
        current_map_mj_to_op = didp_sets['map_mj_to_op']

        # 3. Create Model
        didp_bundle = creation_of_didp_model_function() # Assumes this function exists
        model, _ = didp_bundle

        # 4. Solve
        t_start = pytime.time()
        solver = m_dp.CABS(model, quiet=True, time_limit=time_limit_seconds)
        solution = solver.search()
        duration = pytime.time() - t_start

        # 5. Process Results
        if solution.is_optimal:
            cost = solution.cost
            status = "True"
        elif solution.cost is not None:
            cost = solution.cost
            status = "False (Time Limit)"
        else:
            cost = float('inf')
            status = "False (No Sol)"
            
        gap = "N/A"
        if best_known_cost is not None and cost != float('inf'):
            try:
                gap_val = ((cost - best_known_cost) / best_known_cost) * 100
                gap = f"{gap_val:.2f}%"
            except: gap = "Error"

        print(f"   -> Cost: {cost} | BKC: {best_known_cost} | Gap: {gap}")
        print(f"   -> Time: {duration:.2f}s | Status: {status}")

        result_entry.update({
            "Cost": cost,
            "Gap to BKC": gap,
            "Nodes Expanded": solution.expanded,
            "Nodes Generated": solution.generated,
            "Running Time (s)": duration,
            "Is Optimal": status,
            "Infeasibility": solution.is_infeasible
        })

    except Exception as e:
        print(f"   -> ERROR: {e}")
        result_entry["Infeasibility"] = str(e)

    # 6. Save Immediately
    df_res = pd.DataFrame([result_entry])
    df_res.to_csv(output_csv_name, mode='a', header=not os.path.exists(output_csv_name), index=False)
    
    gc.collect()

print("\n" + "="*50)
print(f"Execution complete. Results saved to {output_csv_name}")


[1/10] Processing: abz5.txt
   -> Cost: 1353.0 | BKC: 1234.0 | Gap: 9.64%
   -> Time: 1800.07s | Status: False (Time Limit)

[2/10] Processing: ft06.txt
   -> Cost: 55.0 | BKC: 55.0 | Gap: 0.00%
   -> Time: 1800.13s | Status: False (Time Limit)

[3/10] Processing: ft10.txt
   -> Cost: 1079.0 | BKC: 930.0 | Gap: 16.02%
   -> Time: 1800.06s | Status: False (Time Limit)

[4/10] Processing: la07.txt
   -> Cost: 944.0 | BKC: 890.0 | Gap: 6.07%
   -> Time: 1800.14s | Status: False (Time Limit)

[5/10] Processing: la21.txt
   -> Cost: 1299.0 | BKC: 1046.0 | Gap: 24.19%
   -> Time: 1800.19s | Status: False (Time Limit)

[6/10] Processing: la26.txt
   -> Cost: 1435.0 | BKC: 1218.0 | Gap: 17.82%
   -> Time: 1800.13s | Status: False (Time Limit)

[7/10] Processing: ta01.txt
   -> Cost: 1662.0 | BKC: 1231.0 | Gap: 35.01%
   -> Time: 1800.21s | Status: False (Time Limit)

[8/10] Processing: ft20.txt
   -> Cost: 1455.0 | BKC: 1165.0 | Gap: 24.89%
   -> Time: 1800.54s | Status: False (Time Limit)

[